# Tool 7: analyze_temporal_change (Grid-Aligned Multi-Temporal Differential Analysis Engine)

This tool performs **pixel-wise differential multi-temporal analysis** on aligned satellite GeoTIFF rasters (e.g., NDVI, NDWI, EVI, or SAR VV/VH backscatter dB). It verifies grid alignment, applies symmetric NoData masking, calculates absolute delta (T2 - T1) and relative percentage shift, applies user-defined or statistical standard-deviation noise thresholds, and exports continuous difference and classified change mask GeoTIFFs.

In [1]:
# 1. Install Dependencies
!pip -q install rasterio numpy pydantic fastmcp


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# 2. Core Imports
import io
import json
import os
import sys
from pathlib import Path
from datetime import datetime, timezone
from typing import Dict, List, Literal, Optional, Tuple, Union, Any

import numpy as np
import rasterio
from rasterio.transform import Affine
from pydantic import BaseModel, Field, field_validator
from fastmcp import FastMCP

In [3]:
# 3. Request Model & Validation
class TemporalChangeRequest(BaseModel):
    raster_before_path: str = Field(..., description="Path to baseline GeoTIFF (T1)")
    raster_after_path: str = Field(..., description="Path to target GeoTIFF (T2)")
    band_selection: Union[int, str] = Field(default=1, description="Band index (1-based) or band name")
    threshold_type: Literal["absolute", "statistical"] = Field(default="absolute", description="Threshold type")
    threshold_value: float = Field(default=0.15, description="Delta threshold value or sigma multiplier")
    relative_change_threshold_percent: Optional[float] = Field(default=None, description="Optional relative shift %")
    mask_encoding: Literal["bipolar_3class", "severity_5class"] = Field(default="bipolar_3class", description="Mask classes")
    output_dir: Optional[str] = Field(default=None, description="Custom output directory")
    generate_difference_raster: bool = Field(default=True, description="Write continuous delta GeoTIFF")
    generate_change_mask: bool = Field(default=True, description="Write classified change mask GeoTIFF")

    @field_validator("raster_before_path", "raster_after_path")
    @classmethod
    def validate_file_exists(cls, value: str) -> str:
        p = Path(value)
        if not p.exists() or not p.is_file():
            raise ValueError(f"Target raster does not exist: '{value}'")
        return str(p.resolve())

In [4]:
# 4. Grid Alignment & Geodesic Utilities
def verify_grid_alignment(src1: rasterio.io.DatasetReader, src2: rasterio.io.DatasetReader, tolerance: float = 1e-5) -> Tuple[bool, dict]:
    same_crs = bool(src1.crs == src2.crs)
    same_dims = bool(src1.width == src2.width and src1.height == src2.height)
    t1 = np.array(src1.transform[:6])
    t2 = np.array(src2.transform[:6])
    max_transform_diff = float(np.max(np.abs(t1 - t2)))
    same_transform = bool(max_transform_diff < tolerance)
    b1 = np.array([src1.bounds.left, src1.bounds.bottom, src1.bounds.right, src1.bounds.top])
    b2 = np.array([src2.bounds.left, src2.bounds.bottom, src2.bounds.right, src2.bounds.top])
    max_bounds_diff = float(np.max(np.abs(b1 - b2)))
    same_bounds = bool(max_bounds_diff < tolerance)
    is_aligned = bool(same_crs and same_dims and same_transform and same_bounds)
    return is_aligned, {
        'is_aligned': is_aligned,
        'crs_match': same_crs,
        'dimensions_match': same_dims,
        'transform_match': same_transform,
        'bounds_match': same_bounds,
        'max_transform_difference': round(max_transform_diff, 8)
    }

def calculate_pixel_area_m2(crs: rasterio.crs.CRS, res_x: float, res_y: float, bounds: rasterio.coords.BoundingBox) -> float:
    if crs and crs.is_geographic:
        lat_mid = (bounds.bottom + bounds.top) / 2.0
        deg_to_m_lat = 111132.954 - 559.822 * np.cos(2 * np.radians(lat_mid)) + 1.175 * np.cos(4 * np.radians(lat_mid))
        deg_to_m_lon = 111412.84 * np.cos(np.radians(lat_mid)) - 93.5 * np.cos(3 * np.radians(lat_mid))
        return float(abs(res_x) * deg_to_m_lon * abs(res_y) * deg_to_m_lat)
    elif crs and crs.is_projected:
        return float(abs(res_x * res_y))
    lat_mid = (bounds.bottom + bounds.top) / 2.0 if bounds else 0.0
    return float(111.320 * np.cos(np.radians(lat_mid)) * abs(res_x) * 110.574 * abs(res_y) * 1e6)

In [5]:
# 5. Import and Execute Core Engine
import sys
sys.path.insert(0, str(Path.cwd().parent))
from Tool_7_analyze_temporal_change.analyze_temporal_change import analyze_temporal_change
print('analyze_temporal_change engine ready!')


analyze_temporal_change engine ready!


In [6]:
# 6. FastMCP Tool Registration
mcp = FastMCP("Tool 7: Grid-Aligned Temporal Change Detection Engine")

@mcp.tool(
    name="analyze_temporal_change",
    description="Perform pixel-wise temporal change detection between two aligned GeoTIFF rasters with thresholding and area accounting."
)
def analyze_temporal_change_mcp(
    raster_before_path: str,
    raster_after_path: str,
    band_selection: Union[int, str] = 1,
    threshold_type: Literal["absolute", "statistical"] = "absolute",
    threshold_value: float = 0.15,
    relative_change_threshold_percent: Optional[float] = None,
    mask_encoding: Literal["bipolar_3class", "severity_5class"] = "bipolar_3class",
    output_dir: Optional[str] = None,
    generate_difference_raster: bool = True,
    generate_change_mask: bool = True
) -> dict:
    try:
        req = TemporalChangeRequest(
            raster_before_path=raster_before_path,
            raster_after_path=raster_after_path,
            band_selection=band_selection,
            threshold_type=threshold_type,
            threshold_value=threshold_value,
            relative_change_threshold_percent=relative_change_threshold_percent,
            mask_encoding=mask_encoding,
            output_dir=output_dir,
            generate_difference_raster=generate_difference_raster,
            generate_change_mask=generate_change_mask
        )
        return analyze_temporal_change(req)
    except ValueError as e:
        return {"status": "error", "error": {"type": "validation_error", "message": str(e)}}
    except Exception as e:
        return {"status": "error", "error": {"type": "service_error", "message": str(e)}}


In [7]:
# 7. Demonstration with Sample Input
req = TemporalChangeRequest(
    raster_before_path="test_runs/sample_indices_t1.tif",
    raster_after_path="test_runs/sample_indices_t2.tif",
    band_selection="NDVI",
    threshold_type="absolute",
    threshold_value=0.15,
    relative_change_threshold_percent=20.0,
    mask_encoding="bipolar_3class",
    output_dir="test_runs",
    generate_difference_raster=True,
    generate_change_mask=True,
)
res = analyze_temporal_change(req)
print('Execution Status:', res.get('status'))
print('Significant Loss Area (km2):', res['change_summary']['significant_decrease']['area_km2'])
print('Significant Gain Area (km2):', res['change_summary']['significant_increase']['area_km2'])
print('Net Delta (km2):', res['change_summary']['net_change_area_km2'])
print('Difference GeoTIFF:', res['generated_products']['difference_raster_path'])
print('Change Mask GeoTIFF:', res['generated_products']['change_mask_path'])


Execution Status: success
Significant Loss Area (km2): 42.3188
Significant Gain Area (km2): 23.6869
Net Delta (km2): -18.6319
Difference GeoTIFF: /Users/dhruvpandey/Desktop/satquery/Tool_7_analyze_temporal_change/test_runs/sample_indices_t1_vs_sample_indices_t2_NDVI_difference.tif
Change Mask GeoTIFF: /Users/dhruvpandey/Desktop/satquery/Tool_7_analyze_temporal_change/test_runs/sample_indices_t1_vs_sample_indices_t2_NDVI_change_mask.tif
